In [2]:
#使用自带的MessageState实现带有消息调用历史的state
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from dotenv import load_dotenv
from IPython.display import Image, display
from operator import add
from langgraph.graph.message import add_messages, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState
from langgraph.checkpoint.memory import InMemorySaver


load_dotenv()

# =================0.先初始化一个模型================
chat_model = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinking":{"type":"disabled"}}
)

# ===============1.定义Node，实现LLM调用===============
def llm_node(state: MessagesState):
    # 获取历史对话消息
    history_messages = state.get("messages")
    # 调用llm，传入message消息历史列表，返回类型是AIMessage
    llm_response = chat_model.invoke(history_messages)
    return {"messages": [llm_response]}

# ===============2.定义Graph===============
graph = StateGraph(MessagesState)
graph.add_node("llm", llm_node)
graph.add_edge(START, "llm")
graph.add_edge("llm", END)
graph_llm = graph.compile(checkpointer = InMemorySaver())

#3.调用graph
config = {"configurable": {"thread_id": "1"}}
res = graph_llm.invoke({"messages": [HumanMessage("我是虎哥，你好")]}, config=config)
print(res)


{'messages': [HumanMessage(content='我是虎哥，你好', additional_kwargs={}, response_metadata={}, id='8b0aec64-c6aa-45ec-9b8e-3ad0357b3ee8'), AIMessage(content='虎哥你好！很高兴认识你！👋\n\n有什么我可以帮你的吗？无论是聊天、解答问题、还是需要帮忙处理什么事情，尽管说！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 9, 'total_tokens': 41, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '36b2e79f-7b2b-4029-97b9-3e1043e8f1cc', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19b6b-5c27-741f-8d26-dbc3b8d35e65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 32, 'total_tokens': 41, 'input_token_details': {'cache_read':

In [ ]:
#测试记忆
res2 = graph_llm.invoke({"messages": [HumanMessage("你好，还记得我是谁吗")]}, config=config)
print(res2)

{'messages': [HumanMessage(content='我是虎哥，你好', additional_kwargs={}, response_metadata={}, id='8b0aec64-c6aa-45ec-9b8e-3ad0357b3ee8'), AIMessage(content='虎哥你好！很高兴认识你！👋\n\n有什么我可以帮你的吗？无论是聊天、解答问题、还是需要帮忙处理什么事情，尽管说！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 9, 'total_tokens': 41, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '36b2e79f-7b2b-4029-97b9-3e1043e8f1cc', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19b6b-5c27-741f-8d26-dbc3b8d35e65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 32, 'total_tokens': 41, 'input_token_details': {'cache_read':

: 